# Preproccessing - Pytorch

The following notebook applies preprocessing techniques using PyTorch. It uses the `California Housing` dataset.

First, import all the libraries that are necessary for this notebook.

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

## Dataset

Let's load the data directly from sklearn as a DataFrame and inspect the first few rows to understand its features.

In [2]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


Let's use the `.info()` method to get a quick summary of our data 

In [3]:
x.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
dtypes: float64(8)
memory usage: 1.3 MB


Using `sklearn`, we split the data into training, validation, and test sets (80/10/10).

In [4]:
x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

Let's verify the split by checking the actual size of each dataset.

In [5]:
print(f"Train set: {len(x_train)}, {len(y_train)}")
print(f"Validation set: {len(x_val)}, {len(y_val)}")
print(f"Test set: {len(x_test)}, {len(y_test)}")

Train set: 16512, 16512
Validation set: 2064, 2064
Test set: 2064, 2064


Now, let's convert our data from pandas DataFrames to PyTorch tensors, which is required for model training.

In [6]:
x_train_tensor = torch.tensor(x_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
x_val_tensor = torch.tensor(x_val.values, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
x_test_tensor = torch.tensor(x_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

## Preproccessing

To improve data quality, we will implement an outlier removal process using PyTorch. This approach follows the same methodology we covered in class, but adapted for tensor operations:

* Calculate the fisrt and third quartiles, and the interquartil range (IQR)
* Define the lower and upper bound for outliers
* Create a copy of the input tensor to avoid modifying the original data
* Cap values (excluiding `Latitude` and `Longitude` features) within the bounds.

In [7]:
class Outliers(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()

        features = x[:, :6]

        Q1 = torch.quantile(features, 0.25, dim=0)
        Q3 = torch.quantile(features, 0.75, dim=0)
        IQR = Q3 - Q1

        self.register_buffer('lower_bound', Q1 - 1.5 * IQR)
        self.register_buffer('upper_bound', Q3 + 1.5 * IQR)

    def forward(self, x) -> torch.Tensor:
        x_capped = x.clone()
        x_capped[:, :6].clip_(self.lower_bound, self.upper_bound) # Generated by AI to cap outliers 

        return x_capped


Now we apply standard scaling:

* Exclude `Latitude` and `Longitude` from scaling.
* Calculate the mean and standard devaition for each feature.
* Normalize each feature by subtracting its mean and dividing by its standard deviation

In [8]:
class ScalingLayer(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()

        features = x[:, :6]

        self.register_buffer('mean', features.mean(dim=0))
        self.register_buffer('std', features.std(dim=0))

    def forward(self, x) -> torch.Tensor:
        return torch.cat([(x[:, :6] - self.mean) / self.std, x[:, 6:]], dim=1) # Generated by AI to apply standard scaling in a single step avoiding Latitude and Longitude

Now we combine both preprocessing steps into a single `PreprocessingLayer` class. This layer will sequentially apply outlier removal and standard scaling to prepare the data for model training.

In [9]:
class PreproccessingLayer(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()

        self.outliers = Outliers(x)
        x_capped = self.outliers(x)

        self.scaler = ScalingLayer(x_capped)

    def forward(self, x) -> torch.Tensor:
        return self.scaler(self.outliers(x))

## Model Training and Evaluation

We implement the training loop function. This function handles model training with early stopping, following the same logic from the Feed Forward Network activity.

In [10]:
def training_loop(model, optimizer, loss_fn, dataloader, epochs=100, delta = 0.005, patience=3):
    model.train()

    epoch_val_loss = []

    for epoch in range(epochs):
        epoch_loss = 0.0

        for X_batch, y_batch in dataloader:
            predictions = model(X_batch)
            loss = loss_fn(predictions, y_batch)

            epoch_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        avg_loss = epoch_loss / len(dataloader)
            
        # Early stopping 
        epoch_val_loss.append(avg_loss)
        if (epoch >= patience):
            if abs(epoch_val_loss[epoch - patience] - epoch_val_loss[epoch]) < delta:  
                if epoch_val_loss[epoch - patience] >= epoch_val_loss[epoch]:
                    print("The model is not improving, early stopping triggered.")
                    break

        print (f'Epoch {epoch+1}, Loss: {avg_loss:.4f}, Loss (MSE): {avg_loss:.4f}')

We define the MSE (Mean Squared Error) loss function, suitable for regression tasks.

In [11]:
loss_fn = nn.MSELoss()

We create a `DataLoader` for the training set, which organizes the data into batches and enables shuffling for better training convergence.

In [12]:
dataset = TensorDataset(x_train_tensor, y_train_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

### First Model

The first model is configured with the following hyperparameters:

Arquitecture:
* Input Layer = 8
* Hidden layer = 64
* Dropout = 0.2
* Hidden layer = 28 
* Output layer = 1

Training Configurations:
* Epochs = 100
* Bacth size = 32
* Optimizer = Adam
* Learning rate = 0.001
* Delta = 0.005
* Patience = 3

In [13]:
class Model_1(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()

        self.preproccessing = PreproccessingLayer(x)
        self.network = nn.Sequential(
            nn.Linear(8, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 28),
            nn.ReLU(),
            nn.Linear(28, 1)
        )

    def forward(self, x) -> torch.Tensor:
        x = self.preproccessing(x)
        return self.network(x)

In [14]:
model_1 = Model_1(x_train_tensor)
optimizer = optim.Adam(model_1.parameters(), lr=0.001)
training_loop(model_1, optimizer, loss_fn, dataloader)

Epoch 1, Loss: 1.4752, Loss (MSE): 1.4752
Epoch 2, Loss: 0.7200, Loss (MSE): 0.7200
Epoch 3, Loss: 0.5956, Loss (MSE): 0.5956
Epoch 4, Loss: 0.5666, Loss (MSE): 0.5666
Epoch 5, Loss: 0.5376, Loss (MSE): 0.5376
Epoch 6, Loss: 0.5222, Loss (MSE): 0.5222
Epoch 7, Loss: 0.5006, Loss (MSE): 0.5006
Epoch 8, Loss: 0.4897, Loss (MSE): 0.4897
Epoch 9, Loss: 0.4847, Loss (MSE): 0.4847
Epoch 10, Loss: 0.4708, Loss (MSE): 0.4708
Epoch 11, Loss: 0.4734, Loss (MSE): 0.4734
Epoch 12, Loss: 0.4638, Loss (MSE): 0.4638
Epoch 13, Loss: 0.4616, Loss (MSE): 0.4616
Epoch 14, Loss: 0.4606, Loss (MSE): 0.4606
Epoch 15, Loss: 0.4556, Loss (MSE): 0.4556
Epoch 16, Loss: 0.4557, Loss (MSE): 0.4557
Epoch 17, Loss: 0.4467, Loss (MSE): 0.4467
Epoch 18, Loss: 0.4456, Loss (MSE): 0.4456
Epoch 19, Loss: 0.4463, Loss (MSE): 0.4463
Epoch 20, Loss: 0.4469, Loss (MSE): 0.4469
The model is not improving, early stopping triggered.


### Second Model

The second model is configured with the following hyperparameters:

Arquitecture:
* Input Layer = 8
* Hidden layer = 128
* Dropout = 0.3
* Hidden layer = 64
* Dropout = 0.2
* Hidden layer = 28 
* Output layer = 1

Training Configurations:
* Epochs = 100
* Bacth size = 32
* Optimizer = Adam
* Learning rate = 0.001
* Delta = 0.001
* Patience = 5

In [15]:
class Model_2(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()

        self.preproccessing = PreproccessingLayer(x)
        self.network = nn.Sequential(
            nn.Linear(8, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 28),
            nn.ReLU(),
            nn.Linear(28, 1)
        )

    def forward(self, x) -> torch.Tensor:
        x = self.preproccessing(x)
        return self.network(x)

In [16]:
model_2 = Model_2(x_train_tensor)
optimizer = optim.Adam(model_2.parameters(), lr=0.001)
training_loop(model_2, optimizer, loss_fn, dataloader, delta=0.001, patience=5)

Epoch 1, Loss: 1.2844, Loss (MSE): 1.2844
Epoch 2, Loss: 0.7388, Loss (MSE): 0.7388
Epoch 3, Loss: 0.6040, Loss (MSE): 0.6040
Epoch 4, Loss: 0.5647, Loss (MSE): 0.5647
Epoch 5, Loss: 0.5394, Loss (MSE): 0.5394
Epoch 6, Loss: 0.5156, Loss (MSE): 0.5156
Epoch 7, Loss: 0.4992, Loss (MSE): 0.4992
Epoch 8, Loss: 0.4885, Loss (MSE): 0.4885
Epoch 9, Loss: 0.4874, Loss (MSE): 0.4874
Epoch 10, Loss: 0.4787, Loss (MSE): 0.4787
Epoch 11, Loss: 0.4680, Loss (MSE): 0.4680
Epoch 12, Loss: 0.4654, Loss (MSE): 0.4654
Epoch 13, Loss: 0.4674, Loss (MSE): 0.4674
Epoch 14, Loss: 0.4600, Loss (MSE): 0.4600
Epoch 15, Loss: 0.4627, Loss (MSE): 0.4627
Epoch 16, Loss: 0.4590, Loss (MSE): 0.4590
Epoch 17, Loss: 0.4516, Loss (MSE): 0.4516
Epoch 18, Loss: 0.4530, Loss (MSE): 0.4530
Epoch 19, Loss: 0.4534, Loss (MSE): 0.4534
Epoch 20, Loss: 0.4480, Loss (MSE): 0.4480
Epoch 21, Loss: 0.4475, Loss (MSE): 0.4475
Epoch 22, Loss: 0.4482, Loss (MSE): 0.4482
Epoch 23, Loss: 0.4444, Loss (MSE): 0.4444
Epoch 24, Loss: 0.44

### Third Model

The third model is configured with the following hyperparameters:

Arquitecture:
* Input Layer = 8
* Hidden layer = 256
* Dropout = 0.4
* Hidden layer = 128
* Dropout = 0.3
* Hidden layer = 64
* Dropout = 0.2
* Hidden layer = 28 
* Output layer = 1

Training Configurations:
* Epochs = 100
* Bacth size = 32
* Optimizer = Adam
* Learning rate = 0.001
* Delta = 0.001
* Patience = 10

In [17]:
class Model_3(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()

        self.preproccessing = PreproccessingLayer(x)
        self.network = nn.Sequential(
            nn.Linear(8, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 28),
            nn.ReLU(),
            nn.Linear(28, 1)
        )

    def forward(self, x) -> torch.Tensor:
        x = self.preproccessing(x)
        return self.network(x)

In [18]:
model_3 = Model_3(x_train_tensor)
optimizer = optim.Adam(model_3.parameters(), lr=0.001)
training_loop(model_3, optimizer, loss_fn, dataloader, delta=0.001, patience=10)

Epoch 1, Loss: 1.3570, Loss (MSE): 1.3570
Epoch 2, Loss: 0.7584, Loss (MSE): 0.7584
Epoch 3, Loss: 0.6429, Loss (MSE): 0.6429
Epoch 4, Loss: 0.5894, Loss (MSE): 0.5894
Epoch 5, Loss: 0.5655, Loss (MSE): 0.5655
Epoch 6, Loss: 0.5448, Loss (MSE): 0.5448
Epoch 7, Loss: 0.5236, Loss (MSE): 0.5236
Epoch 8, Loss: 0.5160, Loss (MSE): 0.5160
Epoch 9, Loss: 0.5061, Loss (MSE): 0.5061
Epoch 10, Loss: 0.5011, Loss (MSE): 0.5011
Epoch 11, Loss: 0.4989, Loss (MSE): 0.4989
Epoch 12, Loss: 0.4915, Loss (MSE): 0.4915
Epoch 13, Loss: 0.4848, Loss (MSE): 0.4848
Epoch 14, Loss: 0.4774, Loss (MSE): 0.4774
Epoch 15, Loss: 0.4745, Loss (MSE): 0.4745
Epoch 16, Loss: 0.4711, Loss (MSE): 0.4711
Epoch 17, Loss: 0.4646, Loss (MSE): 0.4646
Epoch 18, Loss: 0.4667, Loss (MSE): 0.4667
Epoch 19, Loss: 0.4596, Loss (MSE): 0.4596
Epoch 20, Loss: 0.4577, Loss (MSE): 0.4577
Epoch 21, Loss: 0.4583, Loss (MSE): 0.4583
Epoch 22, Loss: 0.4542, Loss (MSE): 0.4542
Epoch 23, Loss: 0.4572, Loss (MSE): 0.4572
Epoch 24, Loss: 0.45

Next, we define the `evaluate()` function to compute and compare the Mean Squared Error (MSE) of all three models on the validation set.

In [19]:
def evaluate(model, x, y, model_name="Model"):
    model.eval()
    with torch.no_grad():
        predictions = model(x)
        mse = loss_fn(predictions, y).item()
        print(f'{model_name} MSE: {mse:.4f}')

The following output displays the validation set MSE for all three models:

In [20]:
print("Validation set evaluation:")
evaluate(model_1, x_val_tensor, y_val_tensor, "Model 1")
evaluate(model_2, x_val_tensor, y_val_tensor, "Model 2")
evaluate(model_3, x_val_tensor, y_val_tensor, "Model 3")

Validation set evaluation:
Model 1 MSE: 0.6230
Model 2 MSE: 0.4952
Model 3 MSE: 0.4402


Based on the validation results, Model 3 achieved the lowest MSE. Let's now evaluate its performance on the test set to confirm generalization.

In [21]:
print("Test set evaluation:")
evaluate(model_3, x_test_tensor, y_test_tensor, "Model 3")

Test set evaluation:
Model 3 MSE: 0.4403
